## **Checking Issues in Dataframes**

Before cleaning, modeling or plotting, a dataframe must be **audited**. This notebook looks for the classic data quality problems: missing values, negative and impossible values, duplicated rows, outliers, identifiers, class imbalance and scale differences.
Every check ends in a number, and the final scorecard and issue log put those numbers together.

**Attention!** This notebook only *detects* issues. Fixing them ( imputation, removal, scaling ) belongs to the preprocessing step, because some fixes must be learned on the training set only.

**Notebooks of this series:** 1. Manual Data Profiling · 2. Automatic Data Profiling · 3. Checking Issues in Dataframes · 4. Graphics & Illustrations · 5. EDA for Time Series

### **00. Import Libs and the DataSet**

In [3]:
#Import Specific Libs
from scipy import stats
from google.colab import files;
from collections import Counter;
from IPython.display import IFrame, display, HTML;

In [4]:
#Import the Others Libs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import re

In [6]:
#Ignore Error Message
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [7]:
#Import Dataframe from Github
df = pd.read_csv('https://raw.githubusercontent.com/jrafa1607/DataScience-ML-AI/main/-%20Datasets/credit_data.csv')

**Attention!** The attributes are renamed exactly as in the *Manual Data Profiling* notebook, so every notebook of the series uses the same names: `Index`, `income`, `age`, `loan` and `result`.

In [11]:
#Define New Attributes Names
df.columns = ['id', 'income', 'age', 'loan', 'result']
df.head(3)

,id,income,age,loan,result
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0


In [12]:
#Change the Name of the Specific Attribute
df = df.rename(columns={df.columns[0]: "Index"})
df.head(3)

,Index,income,age,loan,result
0,1,66155.925095,59.017015,8106.532131,0
1,2,34415.153966,48.117153,6564.745018,0
2,3,57317.170063,63.108049,8020.953296,0


In [20]:
#Define the Numerical Attributes and the Target
num_cols = ['income', 'age', 'loan']
target = 'result'

### **01. Missing Values**

In [27]:
#Check for the Null Values
print(df.isnull().sum())

Index     0
income    0
age       3
loan      0
result    0
dtype: int64


In [28]:
#Percentage of Missing Values
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_%': (df.isnull().mean() * 100).round(2)
})
missing

,missing_count,missing_%
Index,0,0.00
income,0,0.00
age,3,0.15
loan,0,0.00
result,0,0.00


In [ ]:
#Rows with Missing Values
df[df.isnull().any(axis=1)]

In [ ]:
#Missing Values Map ( Each Line = One Attribute, Dark = Missing )
plt.figure(figsize=(10, 2.5))
sns.heatmap(df.isnull().T, cbar=False, cmap='binary', xticklabels=False)
plt.title('Missing Values Map')
plt.xlabel('Rows')
plt.tight_layout()
plt.show()

In [ ]:
#Do the Rows with Missing Values Look Different? ( Missing Data Mechanism )
# MCAR: the missingness is unrelated to anything | MAR: related to other attributes | MNAR: related to the missing value itself ( cannot be tested )
# Attention! With very few missing rows these tests have almost no power: use them as a hint, never as proof
for col in df.columns[df.isnull().any()]:
    is_missing = df[col].isnull()
    print(f'Attribute: {col} | rows with missing values: {is_missing.sum()}')
    rows = []
    for other in [c for c in num_cols + [target] if c != col]:
        when_present = df.loc[~is_missing, other].dropna()
        when_missing = df.loc[is_missing, other].dropna()
        _, p_value = stats.mannwhitneyu(when_present, when_missing)
        rows.append({
            'compared_attribute': other,
            'median_when_present': when_present.median(),
            'median_when_missing': when_missing.median(),
            'p_value': p_value
        })
    print(pd.DataFrame(rows).set_index('compared_attribute').round(4), '\n')

In [ ]:
#Missing Rate by Target Class ( A Hint of MNAR or Biased Data Collection )
df.assign(has_missing=df.isnull().any(axis=1)).groupby(target)['has_missing'].agg(rows_with_missing='sum', missing_rate='mean').round(4)

In [ ]:
#Disguised Missing Values ( Placeholders that Are Not NaN )
# Missing values are often hidden behind placeholders ( 0, -1, 99, 999, 9999 ... ) or behind a suspiciously frequent value
placeholders = [0, -1, -9, 99, 999, 9999, -999, -9999]
pd.DataFrame({
    'placeholder_candidates': {col: int(df[col].isin(placeholders).sum()) for col in num_cols},
    'most_frequent_value_%': {col: round(df[col].value_counts(normalize=True).iloc[0] * 100, 2) for col in num_cols}
})

### **02. Negative and Invalid Values**

In [29]:
#Search for Negative Values
print(df[df['age'] < 0])

    Index        income        age         loan  result
15     16  50501.726689 -28.218361  3977.287432       0
21     22  32197.620701 -52.423280  4244.057136       0
26     27  63287.038908 -36.496976  9595.286289       0


In [30]:
#Second Example (Query) for Negative Values
df.query("age < 0")

,Index,income,age,loan,result
15,16,50501.726689,-28.218361,3977.287432,0
21,22,32197.620701,-52.423280,4244.057136,0
26,27,63287.038908,-36.496976,9595.286289,0


In [ ]:
#Count Negative and Zero Values per Numerical Attribute
pd.DataFrame({
    'negatives': (df[num_cols] < 0).sum(),
    'zeros': (df[num_cols] == 0).sum(),
    'negatives_%': ((df[num_cols] < 0).mean() * 100).round(2)
})

In [ ]:
#Business Rules Validation ( Valid Range per Attribute )
# Each rule returns True for VALID values | change the limits according to the business rules of your problem
rules = {
    'income': lambda s: s > 0,
    'age': lambda s: s.between(18, 100),
    'loan': lambda s: s >= 0,
    'result': lambda s: s.isin([0, 1])
}

validity = pd.DataFrame({
    col: {'checked': int(df[col].notnull().sum()),
          'invalid': int((~rule(df[col]) & df[col].notnull()).sum())}
    for col, rule in rules.items()
}).T
validity['invalid_%'] = (validity['invalid'] / validity['checked'] * 100).round(2)
validity

In [ ]:
#Investigate the Invalid Ages Before Fixing Them
# A negative age may be a sign error ( abs() recovers it ) or a corrupted record ( drop or impute ): the cause decides the fix
invalid_age = df[~df['age'].between(18, 100) & df['age'].notnull()]
invalid_age.assign(abs_age=invalid_age['age'].abs())

### **03. Duplicated Rows**

In [31]:
#Check for Duplicated Rows
print('Duplicated rows ( all attributes ):', df.duplicated().sum())
print('Duplicated rows ( ignoring Index ):', df.drop(columns='Index').duplicated().sum())
print('Duplicated Index values:', df['Index'].duplicated().sum())

Duplicated rows ( all attributes ): 0
Duplicated rows ( ignoring Index ): 0
Duplicated Index values: 0


In [ ]:
#Near-Duplicates ( Rows that Are Equal After Rounding )
# Exact duplicates are easy; near-duplicates ( the same record typed twice with tiny differences ) need a tolerance
rounded = df.drop(columns='Index').round({'income': -2, 'age': 0, 'loan': -1})
print('Near-duplicated rows ( income to 100, age to 1 year and loan to 10 ):', rounded.duplicated().sum())

### **04. Outliers**

In [ ]:
#Outlier Detection Functions ( IQR, Z-Score and Modified Z-Score )
# IQR: outside [ Q1 - 1.5 x IQR , Q3 + 1.5 x IQR ] | Z-Score: |z| > 3 ( assumes a bell-shaped distribution )
# Modified Z-Score: uses the median and the MAD, so the outliers cannot hide themselves ( |mz| > 3.5 )
def flag_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))

def flag_zscore(s, threshold=3):
    return ((s - s.mean()) / s.std()).abs() > threshold

def flag_modified_z(s, threshold=3.5):
    median = s.median()
    mad = (s - median).abs().median()
    return (0.6745 * (s - median) / mad).abs() > threshold

methods = {'IQR': flag_iqr, 'Z-Score': flag_zscore, 'Modified Z-Score': flag_modified_z}
outliers = pd.DataFrame({
    name: {col: int(func(df[col].dropna()).sum()) for col in num_cols}
    for name, func in methods.items()
})
outliers

In [ ]:
#Boxplots of the Numerical Attributes
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f'{col.capitalize()} ( IQR outliers: {outliers.loc[col, "IQR"]} )')
plt.tight_layout()
plt.show()

In [ ]:
#Rows Flagged as Outliers ( Age by the IQR Rule )
# Compare with the invalid ages found in section 02: an invalid value is usually also a statistical outlier
df[flag_iqr(df['age'].dropna()).reindex(df.index, fill_value=False)]

In [ ]:
#Impact of the Outliers on the Statistics ( Age With vs Without Outliers )
age_values = df['age'].dropna()
pd.DataFrame({
    'with_outliers': age_values.describe(),
    'without_outliers': age_values[~flag_iqr(age_values)].describe()
}).round(2)

**Outlier or error?** An *error* ( impossible value ) must be fixed or removed. A *real extreme* ( a very rich client ) must be **kept**, because it is exactly the kind of case the model will meet in production; if it hurts the model, prefer robust techniques ( median, `RobustScaler`, tree models, log transformation ) instead of deleting rows.

### **05. Identifiers, Class Imbalance and Scale**

In [ ]:
#Identifier-Like and Constant Attributes
# An identifier ( unique integers ) has no predictive meaning and can leak the row order | a constant attribute has no information at all
unique_values = df.nunique()
pd.DataFrame({
    'unique_values': unique_values,
    'unique_%': (unique_values / len(df) * 100).round(2),
    'constant': unique_values <= 1,
    'identifier_like': [df[c].is_unique and pd.api.types.is_integer_dtype(df[c]) for c in df.columns]
})

In [ ]:
#Class Imbalance ( Target Check )
counts = df[target].value_counts()
print(f'Majority class: {counts.idxmax()} ( {counts.max()} rows )')
print(f'Minority class: {counts.idxmin()} ( {counts.min()} rows )')
print(f'Imbalance ratio: {counts.max() / counts.min():.1f} : 1')
print(f'A model that always predicts the majority class reaches {counts.max() / counts.sum() * 100:.2f}% accuracy ( the baseline to beat )')

In [ ]:
#Scale Check ( Do the Attributes Live on Different Scales? )
# Distance-based models ( KNN, SVM, K-Means ) and gradient descent are sensitive to the scale of the attributes
scale = df[num_cols].agg(['min', 'max', 'mean', 'std']).T
scale['range'] = scale['max'] - scale['min']
scale['range_vs_smallest'] = (scale['range'] / scale['range'].min()).round(1)
scale.round(2)

### **06. Data Quality Scorecard and Issue Log**

In [ ]:
#Data Quality Scorecard ( Completeness, Validity, Uniqueness and Outlier-Free )
# Each dimension goes from 0 to 100 ( higher is better ) and can be tracked before and after the cleaning
completeness = (1 - df.isnull().sum().sum() / df.size) * 100
validity_score = (1 - validity['invalid'].sum() / validity['checked'].sum()) * 100
uniqueness = (1 - df.duplicated().mean()) * 100
outlier_free = (1 - outliers['IQR'].sum() / df[num_cols].notnull().sum().sum()) * 100

scorecard = pd.Series({
    'Completeness ( no missing cells )': completeness,
    'Validity ( values inside the business rules )': validity_score,
    'Uniqueness ( no duplicated rows )': uniqueness,
    'Outlier-free ( values inside the IQR fences )': outlier_free
}).round(2)
print(scorecard.to_string())
print(f'\nOverall quality score: {scorecard.mean():.2f} / 100')

In [ ]:
#Issue Log ( What Was Found and What To Do )
issue_log = pd.DataFrame([
    ('Missing values', ', '.join(df.columns[df.isnull().any()]), int(df.isnull().sum().sum()), 'Impute ( median or model ) after the train / test split'),
    ('Negative values', ', '.join(c for c in num_cols if (df[c] < 0).any()), int((df[num_cols] < 0).sum().sum()), 'Investigate the cause: fix the sign, drop or impute'),
    ('Values outside the business rules', ', '.join(validity.index[validity['invalid'] > 0]), int(validity['invalid'].sum()), 'Treat as missing and impute'),
    ('Duplicated rows', 'all', int(df.duplicated().sum()), 'Drop the duplicates, keeping the first'),
    ('Outliers ( IQR )', ', '.join(outliers.index[outliers['IQR'] > 0]), int(outliers['IQR'].sum()), 'Separate errors from real extremes before acting'),
    ('Identifier attribute', 'Index', 1, 'Exclude from the features'),
    ('Class imbalance', target, round(counts.max() / counts.min(), 1), 'Stratified split, class weights, resampling and proper metrics ( F1, PR-AUC )'),
    ('Different scales', ', '.join(num_cols), len(num_cols), 'Standardize or normalize ( fit on the training set only )')
], columns=['issue', 'attributes', 'how_many', 'suggested_action'])
issue_log

### **From Detection to Treatment**

The issue log is the bridge to the next step: **Data Preprocessing** ( imputation, negative values, scaling, encoding ) and **Split Dataset in Train and Test** ( stratified and time-aware splits ).
Re-run the scorecard after the cleaning: the score should go up, and any dimension that does not improve shows that the treatment did not work.